# func

In [ ]:
import numpy as np
import scipy.signal as signal
import soundfile as sf

fs = 16000       # サンプリング周波数
duration = 1.0   # 各母音の長さ（秒）
f0_male = 120    # 男性の基本周波数
f0_female = 200  # 女性の基本周波数

# 男性の母音フォルマント値（Hz, BW, Gain）
formants_male = {
    "a": [(730, 80, 1.0), (1090, 90, 1.0), (2440, 120, 1.0)],
    "i": [(270, 60, 1.0), (2290, 100, 1.0), (3010, 120, 1.0)],
    "u": [(300, 70, 1.0), (870, 80, 1.0), (2240, 110, 1.0)],
    "e": [(530, 70, 1.0), (1840, 90, 1.0), (2480, 120, 1.0)],
    "o": [(570, 80, 1.0), (840, 90, 1.0), (2410, 120, 1.0)],
}

# 女性のフォルマントは男性の値を全体的に25%高くする（ゲインはそのまま）
formants_female = {
    v: [(int(f*1.25), bw, g) for f, bw, g in formants_male[v]] for v in formants_male
}

def rosenberg_pulse(period, open_fraction=0.6, fs=16000):
    """
    period: サンプル数（1周期）
    open_fraction: 開放期の割合 (0..1)
    出力: one-period waveform (length=period)
    Rosenberg model: linear rise to peak then cosine-ish fall
    """
    N = period
    No = max(1, int(open_fraction * N))
    Nc = N - No
    t = np.arange(N)
    pulse = np.zeros(N)
    # rising (0..1) linear
    pulse[:No] = (t[:No] / (No - 1 + 1e-12))
    # falling (cosine shaped) from 1 -> 0 over Nc samples
    if Nc > 0:
        tt = np.arange(Nc)
        pulse[No:] = 0.5 * (1 + np.cos(np.pi * (tt / (Nc))))
    # normalize energy
    pulse -= np.mean(pulse)
    pulse /= np.max(np.abs(pulse)) + 1e-12
    return pulse

def impulse(f0, duration, fs):
    """有声音励振（インパルス列）"""
    t = np.linspace(0, duration, int(duration * fs), endpoint=False)
    sig = np.zeros_like(t)
    step = int(fs / f0)
    sig[::step] = 1.0
    return sig

def sqr(t, f):
    return np.sign(np.sin(2 * np.pi * f * t))

def resonator(f, bw, gain, fs):
    """フォルマント共振器フィルタ係数（ゲイン付き）"""
    r = np.exp(-np.pi * bw / fs)
    theta = 2 * np.pi * f / fs
    b = [gain * (1 - r)]
    a = [1, -2 * r * np.cos(theta), r**2]
    return b, a

def synthesize_vowel(formants, f0, duration, fs):
    """フォルマント合成で母音を生成"""
    exc = impulse(f0, duration, fs)
    y = exc.copy()
    for f, bw, g in formants:
        b, a = resonator(f, bw, g, fs)
        y = signal.lfilter(b, a, y)
    return y / np.max(np.abs(y))

# 各母音を合成・保存
for vowel in formants_male.keys():
    # 男性版
    y_m = synthesize_vowel(formants_male[vowel], f0_male, duration, fs)
    sf.write(f"male_{vowel}.wav", y_m, fs)

    # 女性版
    y_f = synthesize_vowel(formants_female[vowel], f0_female, duration, fs)
    sf.write(f"female_{vowel}.wav", y_f, fs)

print("男性・女性版の /a, i, u, e, o/ を wav に保存しました。（ゲイン調整対応）")
